In [0]:
from pyspark.sql.functions import col, lit

In [0]:
TARGET_COLS = [
    "Season","Div","Date","Time","HomeTeam","AwayTeam",
    "FTHG","FTAG","FTR",
    "HTHG","HTAG","HTR",
    "Referee",
    "HS","AS","HST","AST",
    "HF","AF","HC","AC",
    "HY","AY","HR","AR",
    "tournament"
]

In [0]:
def normalize_column_names(df):
    for c in df.columns:
        clean = c.replace("ï»¿", "").strip()
        if c != clean:
            df = df.withColumnRenamed(c, clean)
    return df

In [0]:
def normalize_football_df(df):
    df = normalize_column_names(df)

    existing = set(df.columns)

    if "season" in existing:
        df = df.withColumn("Season", col("season"))
    else:
        df = df.withColumn("Season", lit(None))

    if "Div" not in existing:
        df = df.withColumn("Div", lit(None))

    for c in TARGET_COLS:
        if c not in df.columns:
            df = df.withColumn(c, lit(None))

    return df.select(TARGET_COLS)


In [0]:
patterns = ["championship","conference","league_1","league_2",
            "premier_league","serie_a","serie_b",
            "la_liga_primera","la_liga_segunda"]

sql_likes = " OR ".join([f"table_name LIKE '%{p}%'" for p in patterns])

tables = spark.sql(f"""
SELECT table_name
FROM workspace.information_schema.tables
WHERE table_schema = 'top_5_european_leagues'
AND ({sql_likes})
""").collect()

In [0]:
from functools import reduce

dfs = []

for t in tables:
    full_name = f"top_5_european_leagues.{t.table_name}"
    print(f"------------------------Loading {full_name}------------------------")
    df = spark.read.table(full_name)
    print("DF BEFORE NORMALIZE NECESARY COLUMNS")
    print(df.columns)
    df_norm = normalize_football_df(df)
    print("DF AFTER NORMALIZE NECESARY COLUMNS")
    print(df_norm.columns)
    dfs.append(df_norm)
    print("")
    print("")
    print("")

final_df = reduce(lambda a,b: a.unionByName(b), dfs)
final_df = final_df.withColumn("Div", lit(col("tournament"))).drop("tournament")

In [0]:
display(final_df.limit(5))